# AXI test 15: does a pair of Betti numbers of the recent history carry what the last 50 commands carry? (CPU only, free Colab) - A SECOND RUN OF A TEST THAT IS ALREADY DONE
This test was already run once, in my workspace, and its result is written in `docs/HISTORY_SHAPE_TEST.md` (a "Result" section below the locked predictions). **The roadmap's claim did not hold in that run:** naming the mode, the two loop numbers got 68.6 % against 93.9 % for the last 50 commands; naming the next command, 28.6 % against 65.4 %. This notebook runs the same program on **another machine**. If the printed fingerprint and the table agree with mine, the result does not depend on my workspace. Nothing here is new science; it is a check.

* **No GPU, no model, no key.** numpy and scikit-learn on the CPU. About 15 to 20 minutes.
* **Do not switch to a GPU.** The default runtime is right.
* Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: check the machine (any CPU line is fine)

In [ ]:
!nproc
!python --version
!python -c "import numpy, sklearn; print('numpy', numpy.__version__, '| scikit-learn', sklearn.__version__)"

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-tests.zip` (the one that comes with this notebook)

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: a tiny trial (about 10 seconds): the program must run and print a table. If it says STOP, send the printout

In [ ]:
import sys, os, json, subprocess, time
sys.path.insert(0, "/content/axi/python")
r = subprocess.run([sys.executable, "-u", "-m", "axi.experiments.histshape", "--quick", "--out", "/content/out_quick.json"], cwd="/content/axi/python", env=dict(os.environ, PYTHONPATH="/content/axi/python"), capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-800:])
assert r.returncode == 0 and "HEADLINE" in r.stdout, "STOP: the tiny trial failed. Send the printout above."
print("\nTINY TRIAL OK. Go on to Cell 4.")

### Cell 4: THE HISTORY-SHAPE TEST, same program and same settings as in my run (3 seeds, 20 walks of 10,000 steps each, 9 views). About 15 to 20 minutes. Lines `seed n view: MODE ..% NEXT ..%` appear as each view finishes

In [ ]:
T0 = time.time(); os.makedirs("/content/out", exist_ok=True)
p = subprocess.Popen([sys.executable, "-u", "-m", "axi.experiments.histshape", "--out", "/content/out/axi_histshape_colab.json"], cwd="/content/axi/python", env=dict(os.environ, PYTHONPATH="/content/axi/python"), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line.rstrip())
p.wait(); assert p.returncode == 0, "STOP: the run failed. Send the red text."
print("minutes:", round((time.time() - T0) / 60, 1))

### Cell 5: compare with my run (the fingerprint of the 60 walks must be the same; the accuracies may differ in the last digits)

In [ ]:
from google.colab import files
mine = {"state": [0.363847, 0.280625], "B_betti": [0.685539, 0.28646], "B_betti3": [0.777247, 0.294724], "C_counts": [0.734796, 0.281318], "C_hist": [0.56258, 0.32105], "D_short": [0.694042, 0.571033], "A_window": [0.939319, 0.653881], "A_eng": [0.828671, 0.288693], "A_all": [0.910028, 0.652384]}
res = json.load(open("/content/out/axi_histshape_colab.json"))
print("fingerprint of the 60 walks:", res["digest"], "| mine: c30005542a6567da", "-> SAME" if res["digest"] == "c30005542a6567da" else "-> DIFFERENT (write it in your message)")
worst = 0.0
for arm, (m_mode, m_next) in mine.items():
    mode = sum(res["runs"][s]["arms"][arm]["mode"] for s in res["runs"]) / len(res["runs"]); nxt = sum(res["runs"][s]["arms"][arm]["next"] for s in res["runs"]) / len(res["runs"])
    worst = max(worst, abs(mode - m_mode), abs(nxt - m_next)); print(f"  {arm:10s} MODE {100 * mode:5.1f} % (mine {100 * m_mode:5.1f} %)   NEXT {100 * nxt:5.1f} % (mine {100 * m_next:5.1f} %)")
print(f"largest difference from my run: {100 * worst:.2f} points"); print("REPRODUCED (within half a point)." if worst < 0.005 else "NOT within half a point of my run: send the file.")
files.download("/content/out/axi_histshape_colab.json")

### Done
File downloaded: `axi_histshape_colab.json`. Upload it to the chat with a screenshot of the comparison lines. If the fingerprint is the same and the largest difference is under half a point, the test is reproduced and nothing else needs doing. Do not paste any keys anywhere.